[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xylem-lab/GEO-ANOM-dashboard/blob/main/notebooks/05_all_counties.ipynb)

# 05 — Every poultry house in Maryland, county by county

**Question.** Run notebook 04's method over every county: how many poultry houses are there, where, how accurate is it where hand tracing exists, and how do county totals compare with the registry and the AWTF report?

**How.** For each county, in order (Eastern Shore first, by share of Maryland's poultry, then the rest): a 2 km tile grid → detector → merge → summary (`geo_anom/task1/county.py`, the same functions notebook 04 uses). Aerial photos are streamed, never stored. Each county's results are saved as soon as it finishes; a county that's already done is skipped, and a half-done county resumes where it stopped. So this notebook can be run, interrupted and run again until everything is done.

**Checked against.** USGS hand tracing (Delmarva counties only), a by-eye sample in counties without hand tracing, MDE permits and the AWTF 2017 bird inventory.

**Outputs.** `Task1_results/county_<name>/` per county and `Task1_results/statewide/` (all buildings as GeoJSON + KMZ, the county table) on the shared drive.
**Runtime — run this in Colab, not on a home connection.** About 10,700 tiles statewide, ~40–60 MB of full-resolution imagery read per tile (≈500 GB in total) from Microsoft's West Europe storage. Measured on the project laptop's home connection: ~0.6 MB/s, i.e. ~4 minutes per tile, days for the state. Colab's data-centre network is far faster. Free Colab sessions end after ~12 h; just Run All again and it resumes. With `RUN_FULL = False` it only summarises counties already done.

In [ ]:
# ── Setup: the same cell works in Google Colab and locally ──────────────────────
import os, sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "main"   # branch, tag or commit to run; the exact commit is recorded in run_meta.json

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                     # results go to the shared drive's Task1_results
    ROOT = Path("/content/GEO-ANOM-dashboard")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/xylem-lab/GEO-ANOM-dashboard", str(ROOT)], check=True)
    # always bring the copy up to date (a reused Colab session keeps an old copy)
    head = lambda: subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    before = head()
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin"], check=True)
    remote = f"origin/{CODE_VERSION}"
    target = remote if subprocess.run(["git", "-C", str(ROOT), "rev-parse", "-q", "--verify", remote],
                                      capture_output=True).returncode == 0 else CODE_VERSION
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", target], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)
    if head() != before and "geo_anom" in sys.modules:   # new code, but the old version is already loaded
        raise RuntimeError("Code updated. Runtime -> Restart session, then Run all again.")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
# reload edited geo_anom/ code automatically, without restarting the kernel
if "google.colab" not in sys.modules:   # Colab's IPython can't load autoreload on Python 3.12+ (no `imp`)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import env, tiles, region, viz
from geo_anom.task1 import county as C
from geo_anom.task1.kmz import write_kmz
from geo_anom.task1.pipeline import run_pipeline
from geo_anom.task1.evaluate import evaluate_run, load_ground_truth
print("running in", "Colab" if IN_COLAB else "local", "| results ->", env.results_root())

## Settings

In [ ]:
COUNTIES = C.all_counties_in_order()   # or C.EASTERN_SHORE, or a list like ["Wicomico", "Worcester"]
RUN_FULL = False        # True: run every county not yet done (hours; resumable); False: summarise what's done
MAX_WORKERS = 12        # photos read in parallel
NEAR_PERMIT_M = 1000
print(len(COUNTIES), "counties:", ", ".join(COUNTIES))

## 1. Progress

In [ ]:
def status(name):
    out = C.county_out(name)
    if C.is_complete(name):
        return "done"
    done = len(list((out / "tiles_done").glob("*.json"))) if (out / "tiles_done").exists() else 0
    return f"partial ({done} tiles)" if done else "not started"

progress_tbl = pd.DataFrame([{"county": n, "tiles": len(region.grid_sites(region.county_boundary(n), n)), "status": status(n)}
                             for n in COUNTIES]).set_index("county")
display(progress_tbl)
print(f"{(progress_tbl.status == 'done').sum()} of {len(progress_tbl)} counties done; "
      f"{progress_tbl.loc[progress_tbl.status != 'done', 'tiles'].sum():,} tiles still to run")

## 2. Run the counties that aren't done

Only with `RUN_FULL = True`. Counties run one after another; a failure in one county is reported and the loop moves on (run again to retry it).

Memory: at most 2 × `MAX_WORKERS` photos are held at once (the progress bar shows RAM), so RAM stays around 1–3 GB whatever the county size. Before 2026-10-26 every downloaded photo stayed in memory until its county finished (~9 GB for Worcester), which is what crashed Colab sessions.

In [ ]:
gt = load_ground_truth()
if RUN_FULL:
    from tqdm.auto import tqdm
    for name in COUNTIES:
        if C.is_complete(name):
            continue
        print(f"\n=== {name} ===")
        try:
            out = C.county_out(name)
            if not (out / "run_meta.json").exists() or len(list((out / "tiles_done").glob("*.json"))) < \
                    len(region.grid_sites(region.county_boundary(name), name)):
                C.run_county(name, stream=True, max_workers=MAX_WORKERS, progress=tqdm)
            C.summarize_county(name, gt=gt, near_m=NEAR_PERMIT_M)
            print(f"{name}: done")
        except Exception as e:
            print(f"{name}: FAILED ({type(e).__name__}: {e}) -- run again to retry")
        finally:   # hand memory back between counties (the model is reloaded per county)
            import gc, torch
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

## 3. The county table

Precision / recall only where USGS hand tracing exists (the Delmarva counties). The bottom row pools the counts, not the percentages.

In [ ]:
done = [n for n in COUNTIES if C.is_complete(n)]
summaries = {n: json.loads((C.county_out(n) / "county_summary.json").read_text()) for n in done}
table = pd.DataFrame([C.summary_row(summaries[n]) for n in done]).set_index("county")
acc = [summaries[n]["accuracy"] for n in done if summaries[n]["accuracy"]]
tp, fp, fn = (sum(a[k] for a in acc) for k in ("tp", "fp", "fn"))
total = table.drop(columns=["precision", "recall", "code"]).sum(numeric_only=True)
total["precision"] = tp / (tp + fp) if acc else None
total["recall"] = tp / (tp + fn) if acc else None
table.loc["ALL DONE COUNTIES"] = total
show = table.copy()
for col in ["precision", "recall"]:
    show[col] = show[col].map(lambda v: f"{v:.0%}" if pd.notna(v) else "n/a (no hand tracing)")
display(show)

## 4. Statewide map

In [ ]:
md_all = region.md_counties().to_crs(region.UTM)
parts = []
for n in done:
    _, _, bld = C.load_county_run(C.county_out(n))
    bb = C.county_buildings(bld, region.county_boundary(n), NEAR_PERMIT_M)
    bb["county"] = n
    parts.append(bb)
allb = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=region.UTM) if parts else gpd.GeoDataFrame(geometry=[], crs=region.UTM)
fig, ax = plt.subplots(figsize=(13, 9))
md_all.plot(ax=ax, color="#f4f6f4", edgecolor="#9aa59a", lw=0.6)
if len(allb):
    allb[allb.near_permit].centroid.plot(ax=ax, color="#2c8c3c", markersize=2, label=f"within {NEAR_PERMIT_M} m of a permit ({allb.near_permit.sum():,})")
    allb[~allb.near_permit].centroid.plot(ax=ax, color="#8e44ad", markersize=3, label=f"farther ({(~allb.near_permit).sum():,})")
ax.set_axis_off(); ax.legend(loc="lower left")
ax.set_title(f"{len(allb):,} poultry houses in {len(done)} counties");

In [ ]:
# Save the statewide layers to the shared drive (one KMZ to open in Google Earth)
sw = env.results_root() / "statewide"
sw.mkdir(parents=True, exist_ok=True)
if len(allb):
    allw = allb.to_crs(4326)
    allw.to_file(sw / "buildings_all_counties.geojson", driver="GeoJSON")
    write_kmz(json.loads(allw.drop(columns=["near_permit"]).to_json(drop_id=True, default=str))["features"], sw / "buildings_all_counties.kmz",
              title=f"GEO-ANOM poultry houses, {len(done)} counties")
table.to_csv(sw / "county_table.csv")
print("saved to", sw)

## 5. Counts against the registry and the AWTF report

In [ ]:
t = table.drop(index="ALL DONE COUNTIES", errors="ignore")
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
t[["houses detected", "hand-traced (2016/17)"]].sort_values("houses detected").plot.barh(ax=axs[0], color=["#2c8c3c", "#ff9f1c"])
axs[0].set_title("Houses: detected (2023) vs hand-traced (2016/17)")
x = t["AWTF MDE birds 2017"].fillna(t["AWTF Census birds 2017"])
axs[1].scatter(x, t["houses detected"], color="#2c8c3c")
for n, (xi, yi) in zip(t.index, zip(x, t["houses detected"])):
    if pd.notna(xi): axs[1].annotate(n, (xi, yi), fontsize=8)
axs[1].set_xlabel("AWTF 2017 birds (MDE CAFO data, else USDA Census)"); axs[1].set_ylabel("houses detected")
axs[1].set_title("Houses detected vs. birds per county (AWTF)")
plt.tight_layout()

## 6. By-eye check where there's no hand tracing

Western counties have no USGS tracing, so the only accuracy check is looking. Up to 8 random detections per county; anything that isn't a poultry house is a false positive.

In [ ]:
west = [n for n in done if summaries[n]["accuracy"] is None]
rng = np.random.default_rng(0)
for n in west:
    bb = allb[allb.county == n]
    if not len(bb):
        print(f"{n}: no buildings detected")
        continue
    sample = bb.iloc[rng.choice(len(bb), size=min(8, len(bb)), replace=False)]
    sites_run = json.loads((C.county_out(n) / "sites.json").read_text())
    viz.location_gallery(sample.centroid, sites_run, buildings=bb, cols=4,
                         titles=[f"{r.length_m:.0f}×{r.width_m:.0f} m" for _, r in sample.iterrows()],
                         suptitle=f"{n}: {len(sample)} of {len(bb)} detections")
    plt.show()

## What this shows

*(Write after the counties have run: statewide total, accuracy on the Eastern Shore, how many houses are far from any permit, what the western-county checks found, and how totals compare with the AWTF report.)*